In [6]:
from google.colab import userdata
from huggingface_hub import login
import sys
import os

os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
!git clone https://$GITHUB_TOKEN@github.com/Constantine1824/TRI-AI-SLM.git
sys.path.append('/content/TRI-AI-SLM')
login(token=userdata.get('HF_TOKEN'))
%cd /content/TRI-AI-SLM

Cloning into 'TRI-AI-SLM'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 87 (delta 20), reused 49 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 59.86 KiB | 3.74 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/TRI-AI-SLM


In [2]:
import importlib
importlib.invalidate_caches()

import importlib.util
print(importlib.util.find_spec('finetune'))

None


In [3]:
#!pip install trl peft
import numpy as np
import pandas as pd
from finetune.trainer import collate_fn, finetune, run_inference
from utils.format import format_train_data, format_test_data

In [7]:
data = pd.read_csv('data/train_qa.csv')
data.head()

,question,topic,care_setting,population,document_id,reference_answer,QuestionId
0,BP readings improved — skip tablets this weekend?,chronic_disease,primary_care,adult,doc_chr_002,No — never stop antihypertensives abruptly wit...,1
1,Pregnant and always tired — common workup?,maternal_health,primary_care,pregnant,doc_mat_003,Screen for anaemia and treat with iron plus di...,2
2,When should someone with flu-like symptoms sta...,infectious_disease,community,general,doc_inf_002,While febrile to reduce spread at school or work.,3
3,How do I manage type 2 diabetes day to day?,chronic_disease,primary_care,adult,doc_chr_001,"Balance meals, stay active, take medicines as ...",4
4,Feverish toddler — paracetamol cautions?,medication_safety,home,child,doc_med_003,Use weight-based doses and avoid exceeding dai...,5


In [8]:
doc = pd.read_csv('data/documents.csv')
doc.head()

,document_id,title,topic,care_setting,population,text,origin,source_url,license
0,doc_chr_001,Type 2 diabetes self-management,chronic_disease,primary_care,adult,Type 2 diabetes management combines balanced m...,synthetic,NaN,CC0-1.0
1,doc_chr_002,Hypertension lifestyle measures,chronic_disease,primary_care,adult,"Lowering dietary salt, maintaining healthy wei...",synthetic,NaN,CC0-1.0
2,doc_chr_003,Asthma action plan basics,chronic_disease,primary_care,child,Children with asthma should use a written acti...,synthetic,NaN,CC0-1.0
3,doc_inf_001,Malaria prevention in endemic areas,infectious_disease,community,general,"Sleep under insecticide-treated nets, eliminat...",synthetic,NaN,CC0-1.0
4,doc_inf_002,Hand hygiene and respiratory etiquette,infectious_disease,community,general,Wash hands with soap for at least twenty secon...,synthetic,NaN,CC0-1.0


In [9]:
data = data.merge(doc[['document_id', 'text']], on='document_id', how='left')
data = data.rename(columns={'text':'context'})
data.head()

,question,topic,care_setting,population,document_id,reference_answer,QuestionId,context
0,BP readings improved — skip tablets this weekend?,chronic_disease,primary_care,adult,doc_chr_002,No — never stop antihypertensives abruptly wit...,1,"Lowering dietary salt, maintaining healthy wei..."
1,Pregnant and always tired — common workup?,maternal_health,primary_care,pregnant,doc_mat_003,Screen for anaemia and treat with iron plus di...,2,Anaemia increases fatigue and adverse birth ou...
2,When should someone with flu-like symptoms sta...,infectious_disease,community,general,doc_inf_002,While febrile to reduce spread at school or work.,3,Wash hands with soap for at least twenty secon...
3,How do I manage type 2 diabetes day to day?,chronic_disease,primary_care,adult,doc_chr_001,"Balance meals, stay active, take medicines as ...",4,Type 2 diabetes management combines balanced m...
4,Feverish toddler — paracetamol cautions?,medication_safety,home,child,doc_med_003,Use weight-based doses and avoid exceeding dai...,5,Use weight-based paediatric paracetamol dosing...


In [ ]:
from datasets import Dataset
test_data = pd.read_csv('/data/test_questions.csv')

train_data = Dataset.from_pandas(data)
test_data = Dataset.from_pandas(test_data)
train_data = train_data.map(format_train_data)
test_data = test_data.map(format_test_data)